# Show2D

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/docs/tutorials/show2d.ipynb)

`Show2D` renders one or many 2D images with contrast control, an FFT toggle, ROIs, line profiles, and a calibrated scale bar. It accepts a NumPy array, a PyTorch tensor, or a quantem `Dataset2d`.

This tutorial uses a real gold HAADF image from the public [`bobleesj/quantem-data`](https://huggingface.co/datasets/bobleesj/quantem-data) Hugging Face dataset. The full image is 4096 by 4096 pixels; `show2d_gold(size="small")` returns a calibrated preview so the notebook starts quickly in the documentation site and in Colab.

```{tip}
Run this exact notebook with the Colab badge above, or [View or download this notebook on GitHub](https://github.com/bobleesj/quantem.widget/blob/main/docs/tutorials/show2d.ipynb). For finished results, use [HTML and file export](widget_export) to export interactive HTML or share a trusted notebook with widget state.
```


In [ ]:
import subprocess
import sys

try:
    import google.colab  # noqa: F401
except Exception:
    pass
else:
    from google.colab import output

    output.enable_custom_widget_manager()
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/bobleesj/quantem.widget.git"],
        check=True,
    )


In [1]:
import numpy as np
import torch
from quantem.core.datastructures import Dataset2d
from quantem.widget import Show2D
from quantem.widget.datasets import show2d_gold

dataset2d = show2d_gold(size="small")
image = dataset2d.array


Source: gold_haadf_npy from Hugging Face
Full image: 4096 x 4096 uint16
Preview: 512 x 512, pixel size 0.1489 nm


## Single image

The HAADF image is wrapped in a quantem `Dataset2d`, so the calibration travels with the array. The widget reads that metadata automatically and draws a physical scale bar without a widget-level pixel-size argument.


In [2]:
Show2D(dataset2d, cmap="inferno")


Show2D(512×512, cmap=inferno)

## A gallery of related views

Pass a list of calibrated `Dataset2d` objects to compare related real-data views side by side. The contrast and zoom controls can be linked across panels.


In [3]:
rng = np.random.default_rng(0)
noise = 0.03 * float(np.std(image)) * rng.standard_normal(image.shape).astype(np.float32)
variant_arrays = [image, np.fliplr(image), image + noise]
variant_names = ["original", "flipped", "noisy"]
variant_datasets = [
    Dataset2d.from_array(arr, sampling=dataset2d.sampling, units=dataset2d.units, name=name)
    for arr, name in zip(variant_arrays, variant_names)
]

Show2D(variant_datasets, labels=variant_names)


Show2D(3×512×512, idx=0, cmap=inferno)

## Page through a reconstruction sweep

Use pages when each step has the same panel layout. This compact torch-generated example is intentionally small so the documentation and Colab version open quickly, while still showing the workflow used for lambda or iteration sweeps.


In [4]:
torch.manual_seed(3)
page_labels = ["lambda 0.01", "lambda 0.03", "lambda 0.10"]
panel_labels = ["raw", "filtered", "residual", "score"]

n = 160
y, x = torch.meshgrid(
    torch.linspace(-1, 1, n),
    torch.linspace(-1, 1, n),
    indexing="ij",
)
centers = torch.tensor([
    [-0.45, -0.35], [-0.05, -0.40], [0.35, -0.30],
    [-0.35, 0.05], [0.05, 0.02], [0.45, 0.10],
    [-0.18, 0.45], [0.28, 0.48],
])
base = torch.zeros((n, n), dtype=torch.float32)
for cy, cx in centers:
    base += torch.exp(-((x - cx) ** 2 + (y - cy) ** 2) / 0.010)
base = base / base.max()
texture = 0.10 * torch.sin(34 * x + 4 * torch.sin(8 * y)) * torch.cos(28 * y)
background = 0.18 * torch.exp(-((x + 0.70) ** 2 + (y - 0.62) ** 2) / 0.18)

pages = []
for page, (lam, denoise, blur_radius, artifact_strength, shift) in enumerate(zip(
    [0.01, 0.03, 0.10],
    [0.10, 0.45, 0.78],
    [0, 3, 7],
    [0.34, 0.16, 0.05],
    [-12, 0, 12],
)):
    shifted_base = torch.roll(base, shifts=(shift, -shift // 2), dims=(0, 1))
    shifted_texture = torch.roll(texture, shifts=(-shift // 2, shift), dims=(0, 1))
    noise = artifact_strength * torch.randn_like(base)
    stripes = artifact_strength * torch.sin((18 + 90 * lam) * x + (10 + 4 * page) * y)
    raw = shifted_base + shifted_texture + background + noise + stripes
    filtered = raw.clone()
    for _ in range(blur_radius):
        filtered = 0.5 * filtered + 0.125 * (
            torch.roll(filtered, 1, 0)
            + torch.roll(filtered, -1, 0)
            + torch.roll(filtered, 1, 1)
            + torch.roll(filtered, -1, 1)
        )
    filtered = (1 - denoise) * raw + denoise * (0.85 * filtered + 0.15 * shifted_base)
    residual = raw - filtered
    score = 2.5 * torch.abs(residual) + lam * shifted_base
    pages.append(torch.stack([raw, filtered, residual, score]))
page_panels = torch.stack(pages).numpy().astype(np.float32)


In [5]:
paged_show2d = Show2D(
    page_panels,
    labels=panel_labels,
    page_labels=page_labels,
    ncols=4,
    cmap="inferno",
    sampling=(0.025, 0.025),
    units="nm",
    link_contrast=False,
)
paged_show2d.star_page(1)
paged_show2d


Show2D(12×160×160, idx=0, cmap=inferno)

## Trigger a fresh render in an existing widget

When a notebook loop produces a new image, keep the same widget object and call `set_image()`. That method is the render trigger: it sends fresh synced image bytes to the browser and resets stale panel-specific state. Mutating the original NumPy array in place is not enough because the frontend only repaints when a synced trait changes.

Use `offline=False` for acquisition-style updates so each replacement travels through the live Jupyter Comm path instead of the saved/offline notebook representation.


In [6]:
live2d = Show2D(dataset2d, labels=["original"], offline=False, cmap="inferno")
live2d


Show2D(512×512, cmap=inferno)

In [7]:
updated_gallery = np.stack([
    image,
    np.flipud(image),
    np.clip(image * 0.85 + 0.15 * float(image.mean()), 0, None),
])

live2d.set_image(
    updated_gallery,
    labels=["original", "flipped up/down", "contrast adjusted"],
)
